### MODEL TRAINING

In [ ]:
# Basic Import
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt 
import seaborn as sns
# Modelling
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.neighbors import KNeighborsRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor,AdaBoostRegressor
from sklearn.svm import SVR
from sklearn.linear_model import LinearRegression, Ridge,Lasso
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error
from sklearn.model_selection import RandomizedSearchCV
from catboost import CatBoostRegressor
from xgboost import XGBRegressor
from sklearn.metrics import f1_score, confusion_matrix, classification_report

import warnings


In [85]:
df = pd.read_csv('data/cleaned_apple_sales.csv')

In [86]:
df.head(5)
df.columns

Index(['year', 'quarter', 'month', 'country', 'region', 'category', 'storage',
       'unit_price_usd', 'discount_pct', 'units_sold', 'revenue_usd',
       'sales_channel', 'payment_method', 'customer_segment',
       'customer_age_group', 'previous_device_os', 'return_status', 'day',
       'day_of_week'],
      dtype='object')

In [87]:
X = df.drop(columns=['return_status'],axis=1)
y = df['return_status']

In [88]:
X.shape

(11500, 18)

In [89]:
y.value_counts()

return_status
Kept         10143
Returned       898
Exchanged      459
Name: count, dtype: int64

In [90]:
# -------------------------------------------------------------------------------------------------------------------------------------------------

# Label Encoding: quarter, customer_age_group, storage, month, day_of_week
# One Hot Encoding: region, category, sales_channel, payment_method, customer_segment, previous_device_os
# Target Encoding: country 
# StandardScaler: year, day, discount_pct, units_sold, unit_price_usd, revenue_usd 

In [91]:
numerical_featuers = X.select_dtypes(exclude='object')
categorical_featuers = X.select_dtypes(include='object')

numerical_cols = [col for col in numerical_featuers.columns.tolist() 
                if col not in ['month', 'day_of_week']]

OHE_featuers = ["region", "category", "sales_channel", "payment_method", "customer_segment", "previous_device_os"]
TE_featuers = ["country"]
LE_featuers = ["quarter", "customer_age_group", "storage", "month", "day_of_week"]


In [92]:
from sklearn.preprocessing import OneHotEncoder, StandardScaler, TargetEncoder, OrdinalEncoder
from sklearn.compose import ColumnTransformer

oh_transformer = OneHotEncoder()
numeric_transformer = StandardScaler()
target_encoder = TargetEncoder()
ordinal_encoder = OrdinalEncoder()

preprocessor = ColumnTransformer([
    ("OneHotEncoder", oh_transformer, OHE_featuers),
    ("StandardScaler", numeric_transformer, numerical_cols),
    ("TargetEncoder", target_encoder, TE_featuers),
    ("OrdinalEncoder", ordinal_encoder, LE_featuers)
])




In [93]:
X = preprocessor.fit_transform(X,y)


In [94]:
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()
y = le.fit_transform(y)  

In [95]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X,y,test_size=0.2,random_state=2)
X_train.shape, X_test.shape

((9200, 52), (2300, 52))

In [96]:


def evaluate_model(true, predicted):
    f1score_weighted = f1_score(true, predicted, average='weighted',zero_division=0)
    f1score_macro = f1_score(true, predicted, average='macro',zero_division= 0 )
    confusionMatrix = confusion_matrix(true, predicted)
    classificationReport = classification_report(true ,predicted)
    return f1score_weighted,f1score_macro,confusion_matrix,classification_report


In [97]:
from sklearn.linear_model import LogisticRegression, RidgeClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier, GradientBoostingClassifier
from sklearn.naive_bayes import GaussianNB, BernoulliNB
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC, LinearSVC
from sklearn.neural_network import MLPClassifier
from xgboost import XGBClassifier
from catboost import CatBoostClassifier

In [98]:
models = {
    "Logistic Regression": LogisticRegression(class_weight='balanced', max_iter=1000),
    "Ridge Classifier": RidgeClassifier(class_weight='balanced'),
    "Decision Tree": DecisionTreeClassifier(class_weight='balanced'),
    "Random Forest": RandomForestClassifier(class_weight='balanced'),
    "Gradient Boosting": GradientBoostingClassifier(),
    "XGBoost": XGBClassifier(),
    "CatBoost": CatBoostClassifier(auto_class_weights='Balanced', verbose=0),
    "Gaussian NB": GaussianNB(),
    "Bernoulli NB": BernoulliNB(),
    "KNN": KNeighborsClassifier(),
    "SVC": SVC(class_weight='balanced', probability=True),
    "LinearSVC": LinearSVC(class_weight='balanced')
}

model_list = []
models_performance = {}

for model_name,model in models.items():
    model.fit(X_train,y_train)

    y_train_pred = model.predict(X_train)
    y_test_pred = model.predict(X_test)

    model_train_f1score_weighted,model_train_f1score_macro,model_train_confusion_matrix,model_train_classification_report = evaluate_model(y_train, y_train_pred) 
    model_test_f1score_weighted,model_test_f1score_macro,model_test_confusion_matrix,model_test_classification_report = evaluate_model(y_test, y_test_pred) 

    models_performance[model_name] = {
        "train" : {
            "weighted_f1": model_train_f1score_weighted,
            "macro_f1": model_train_f1score_macro,
            "confusion_matrix": model_train_confusion_matrix,
            "classification_report": model_train_classification_report
        },
        "test" : {
            "weighted_f1": model_test_f1score_weighted,
            "macro_f1": model_test_f1score_macro,
            "confusion_matrix": model_test_confusion_matrix,
            "classification_report": model_test_classification_report
        }
    }
    for model_name, performance in models_performance.items():
        print(f"{model_name} -> Train Weighted F1: {performance['train']['weighted_f1']:.4f} | Test Weighted F1: {performance['test']['weighted_f1']:.4f}")

Logistic Regression -> Train Weighted F1: 0.4654 | Test Weighted F1: 0.4572
Logistic Regression -> Train Weighted F1: 0.4654 | Test Weighted F1: 0.4572
Ridge Classifier -> Train Weighted F1: 0.4542 | Test Weighted F1: 0.4476
Logistic Regression -> Train Weighted F1: 0.4654 | Test Weighted F1: 0.4572
Ridge Classifier -> Train Weighted F1: 0.4542 | Test Weighted F1: 0.4476
Decision Tree -> Train Weighted F1: 1.0000 | Test Weighted F1: 0.7847


c:\anaconda3\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\anaconda3\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\anaconda3\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])


Logistic Regression -> Train Weighted F1: 0.4654 | Test Weighted F1: 0.4572
Ridge Classifier -> Train Weighted F1: 0.4542 | Test Weighted F1: 0.4476
Decision Tree -> Train Weighted F1: 1.0000 | Test Weighted F1: 0.7847
Random Forest -> Train Weighted F1: 1.0000 | Test Weighted F1: 0.8269
Logistic Regression -> Train Weighted F1: 0.4654 | Test Weighted F1: 0.4572
Ridge Classifier -> Train Weighted F1: 0.4542 | Test Weighted F1: 0.4476
Decision Tree -> Train Weighted F1: 1.0000 | Test Weighted F1: 0.7847
Random Forest -> Train Weighted F1: 1.0000 | Test Weighted F1: 0.8269
Gradient Boosting -> Train Weighted F1: 0.8308 | Test Weighted F1: 0.8278


c:\anaconda3\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\anaconda3\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\anaconda3\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])


Logistic Regression -> Train Weighted F1: 0.4654 | Test Weighted F1: 0.4572
Ridge Classifier -> Train Weighted F1: 0.4542 | Test Weighted F1: 0.4476
Decision Tree -> Train Weighted F1: 1.0000 | Test Weighted F1: 0.7847
Random Forest -> Train Weighted F1: 1.0000 | Test Weighted F1: 0.8269
Gradient Boosting -> Train Weighted F1: 0.8308 | Test Weighted F1: 0.8278
XGBoost -> Train Weighted F1: 0.9845 | Test Weighted F1: 0.8269
Logistic Regression -> Train Weighted F1: 0.4654 | Test Weighted F1: 0.4572
Ridge Classifier -> Train Weighted F1: 0.4542 | Test Weighted F1: 0.4476
Decision Tree -> Train Weighted F1: 1.0000 | Test Weighted F1: 0.7847
Random Forest -> Train Weighted F1: 1.0000 | Test Weighted F1: 0.8269
Gradient Boosting -> Train Weighted F1: 0.8308 | Test Weighted F1: 0.8278
XGBoost -> Train Weighted F1: 0.9845 | Test Weighted F1: 0.8269
CatBoost -> Train Weighted F1: 0.9926 | Test Weighted F1: 0.8154
Logistic Regression -> Train Weighted F1: 0.4654 | Test Weighted F1: 0.4572
Ridge

c:\anaconda3\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\anaconda3\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\anaconda3\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\anaconda3\Lib\site-packages\sklearn\metrics\_classification.p

Logistic Regression -> Train Weighted F1: 0.4654 | Test Weighted F1: 0.4572
Ridge Classifier -> Train Weighted F1: 0.4542 | Test Weighted F1: 0.4476
Decision Tree -> Train Weighted F1: 1.0000 | Test Weighted F1: 0.7847
Random Forest -> Train Weighted F1: 1.0000 | Test Weighted F1: 0.8269
Gradient Boosting -> Train Weighted F1: 0.8308 | Test Weighted F1: 0.8278
XGBoost -> Train Weighted F1: 0.9845 | Test Weighted F1: 0.8269
CatBoost -> Train Weighted F1: 0.9926 | Test Weighted F1: 0.8154
Gaussian NB -> Train Weighted F1: 0.7998 | Test Weighted F1: 0.7992
Bernoulli NB -> Train Weighted F1: 0.8266 | Test Weighted F1: 0.8269
Logistic Regression -> Train Weighted F1: 0.4654 | Test Weighted F1: 0.4572
Ridge Classifier -> Train Weighted F1: 0.4542 | Test Weighted F1: 0.4476
Decision Tree -> Train Weighted F1: 1.0000 | Test Weighted F1: 0.7847
Random Forest -> Train Weighted F1: 1.0000 | Test Weighted F1: 0.8269
Gradient Boosting -> Train Weighted F1: 0.8308 | Test Weighted F1: 0.8278
XGBoost 

c:\anaconda3\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\anaconda3\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\anaconda3\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\anaconda3\Lib\site-packages\sklearn\metrics\_classification.p